# SimpleFold nanobody structures — 6 hand-picked cases

Folds only the six targets chosen from the 597-target benchmark, superimposes each prediction
on its crystal structure, and writes both as PDB with **per-atom error in the B-factor column**
so a viewer can colour by error directly.

Sampling is stochastic and seeded once per process, so folding 6 targets draws different noise
than folding 597 did. Each structure is therefore re-scored here, and both numbers are reported.

In [ ]:
import os, sys, json, time, glob, shutil, subprocess
import numpy as np

FORK, BRANCH = 'https://github.com/omar-A-hassan/ml-simplefold', 'fix-multichain-fasta-parsing'
MODEL, TAU, STEPS, SEED = 'simplefold_100M', '0.01', '500', '12345'
WORK, SCRATCH = '/kaggle/working', '/tmp/sf'
os.makedirs(SCRATCH, exist_ok=True)
OUT = f'{WORK}/structures'; os.makedirs(OUT, exist_ok=True)

import torch
n, cap = torch.cuda.device_count(), torch.cuda.get_device_capability(0)
assert cap >= (7, 0), f'{torch.cuda.get_device_name(0)} is sm_{cap[0]}{cap[1]}; need sm_70+'
print('GPU:', torch.cuda.get_device_name(0), f'(x{n})')

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, _ds
TG = {t['instance']: t for t in json.load(open(_ds[0]))}

PICKS = [
 ('8EMYL',   'pdb_00008emy_L_+', 'best CDR-H3 in the benchmark -- what success looks like'),
 ('9G13A',   'pdb_00009g13_A_+', 'worst case: whole domain misfolded (also worst framework)'),
 ('7B5GF',   'pdb_00007b5g_F_+', 'the median nanobody -- representative case'),
 ('7Z1DFFF', 'pdb_00007z1d_FFF_+','20-residue loop solved to 0.73 A -- hard case done well'),
 ('7TPRD',   'pdb_00007tpr_D_+', 'framework fine, loop 13 A off -- biggest chain-vs-fw split'),
 ('7FATB',   'pdb_00007fat_B_+', 'longest CDR-H3 in the set (26 residues)'),
]
# instance ids were guessed from tid; resolve them properly against the dataset
by_tid = {f"{t['pdb']}{t['H']}": t for t in TG.values()}
PICKS = [(tid, by_tid[tid]['instance'], why) for tid, _, why in PICKS]
assert len(PICKS) == 6
print('\n'.join(f'{t:<9} {w}' for t, _, w in PICKS))

## Install + fp16 patch

In [ ]:
def must(c, what):
    p = subprocess.run(c, shell=True, text=True, capture_output=True)
    if p.returncode: raise RuntimeError(f'{what} failed:\n{(p.stdout+p.stderr)[-1500:]}')

must(f'git clone -q --depth 1 -b {BRANCH} {FORK} {SCRATCH}/repo', 'clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo', 'install simplefold')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git', 'install esm')
must(f'{sys.executable} -m pip install -q biotite', 'install biotite')
assert 'chr(ord("A") + i)' in open(f'{SCRATCH}/repo/src/simplefold/utils/fasta_utils.py').read()

INF = f'{SCRATCH}/repo/src/simplefold/inference.py'
s = open(INF).read()
anchor = '        esm_model = esm_model.to(device)'
assert s.count(anchor) == 1
open(INF, 'w').write(s.replace(anchor, '        esm_model = esm_model.half().to(device)'))
print('fp16 patch applied')

## Fold the six

In [ ]:
fadir = f'{SCRATCH}/fa'; os.makedirs(fadir, exist_ok=True)
for tid, inst, _ in PICKS:
    open(f'{fadir}/{tid}.fasta', 'w').write(f">A|{tid}\n{TG[inst]['hseq']}\n")

t0 = time.time()
p = subprocess.run(['simplefold','--simplefold_model',MODEL,'--backend','torch',
                    '--ckpt_dir',f'{SCRATCH}/artifacts','--num_steps',STEPS,'--tau',TAU,
                    '--nsample_per_protein','1','--seed',SEED,
                    '--fasta_path',fadir,'--output_dir',f'{SCRATCH}/out'],
                   capture_output=True, text=True)
if p.returncode: raise RuntimeError((p.stdout + p.stderr)[-3000:])
cifs = glob.glob(f'{SCRATCH}/out/**/*.cif', recursive=True)
print(f'{len(cifs)} structures in {time.time()-t0:.0f}s')

## Score, superimpose, write

Superposition is **framework-only**: the scaffold is pinned so loop error is shown honestly
rather than being spread across the whole domain by a global fit.

In [ ]:
import biotite.structure.io.pdbx as pdbx

def kabsch(P, Q):
    Pm, Qm = P.mean(0), Q.mean(0)
    V, _, W = np.linalg.svd((P - Pm).T @ (Q - Qm))
    D = np.diag([1, 1, np.sign(np.linalg.det(V @ W))])
    return V @ D @ W, Pm, Qm

def write_pdb(path, names, resnames, resids, xyz, bfac, chain='A'):
    with open(path, 'w') as f:
        for i, (nm, rn, ri, c, b) in enumerate(zip(names, resnames, resids, xyz, bfac), 1):
            if not np.isfinite(c[0]): continue
            f.write(f'ATOM  {i:>5} {nm:<4}{rn:>4} {chain}{ri:>4}    '
                    f'{c[0]:>8.3f}{c[1]:>8.3f}{c[2]:>8.3f}  1.00{b:>6.2f}          {nm[0]:>2}\n')
        f.write('END\n')

ATOMS = ['N','CA','C','CB']
report = []
for tid, inst, why in PICKS:
    t = TG[inst]; Ln = len(t['hseq'])
    g = glob.glob(f'{SCRATCH}/out/**/*{tid}*.cif', recursive=True)
    assert len(g) == 1, (tid, g)
    a = pdbx.get_structure(pdbx.CIFFile.read(g[0]), model=1)
    u = np.unique(a.res_id)
    assert len(np.unique(a.chain_id)) == 1 and len(u) == Ln and u.max()-u.min()+1 == Ln, tid
    rid = a.res_id - a.res_id.min()

    pred = np.full((Ln, 4, 3), np.nan)
    for k, an in enumerate(ATOMS):
        m = a.atom_name == an
        pred[rid[m], k] = a.coord[m]
    ref = np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']], float)

    reg = {}
    for k in ('cdr1','cdr2','cdr3'):
        m = np.zeros(Ln, bool); lo, hi = t[k]; m[lo:hi] = True; reg[k] = m
    fw = ~(reg['cdr1'] | reg['cdr2'] | reg['cdr3'])
    ok = np.isfinite(ref[...,0]) & np.isfinite(pred[...,0])

    Rot, Pm, Qm = kabsch(pred[ok & fw[:,None]], ref[ok & fw[:,None]])
    rmsd = lambda s: float(np.sqrt((( (pred[s]-Pm)@Rot+Qm - ref[s])**2).sum(-1).mean()))
    sc = {k: round(rmsd(ok & reg[k][:,None]), 2) for k in ('cdr1','cdr2','cdr3')}
    sc['framework'] = round(rmsd(ok & fw[:,None]), 2)

    # per-residue error -> B-factor, so a viewer can colour by accuracy
    dev = np.full(Ln, 0.0)
    d = np.linalg.norm((pred-Pm)@Rot+Qm - ref, axis=-1)
    for i in range(Ln):
        v = d[i][np.isfinite(d[i])]
        dev[i] = v.mean() if len(v) else 0.0

    allx = (a.coord - Pm) @ Rot + Qm                       # transform EVERY atom, side chains too
    write_pdb(f'{OUT}/{tid}_pred.pdb', a.atom_name, a.res_name, a.res_id, allx, dev[rid])
    rn = [t['hseq'][i] for i in range(Ln)]
    write_pdb(f'{OUT}/{tid}_ref.pdb',
              [an for i in range(Ln) for an in ATOMS],
              [rn[i] for i in range(Ln) for _ in ATOMS],
              [i+1 for i in range(Ln) for _ in ATOMS],
              ref.reshape(-1,3), np.repeat(dev, 4))
    report.append({'tid': tid, 'why': why, 'cdr3_len': int(t['cdr3'][1]-t['cdr3'][0]),
                   'L': Ln, 'rescored_fw_aligned': sc, 'max_res_dev': round(float(dev.max()),2)})
    print(f"{tid:<9} cdr3 {sc['cdr3']:>6.2f}  fw {sc['framework']:>5.2f}   {why}")

json.dump(report, open(f'{OUT}/viz_report.json','w'), indent=1)
shutil.rmtree(f'{SCRATCH}/out', ignore_errors=True)
print('\nwrote', len(os.listdir(OUT)), 'files;',
      round(sum(os.path.getsize(f'{OUT}/{f}') for f in os.listdir(OUT))/1e6, 2), 'MB')